# Musanga V3 — Notebook 01: Reference data and spatial cross-validation

Goal: create the best full-ROI UAV Musanga probability maps for satellite fractional-cover mapping. This notebook prepares reference geometry, uncertainty policy, spatial folds and candidate windows; it does not train a model or rasterize multi-GB orthomosaics. Four forest groups: Baego, MOS (three mosaics, one group), Yangambi and Yoko. Primary validation: within-site spatial held-out exhaustive circles; final production refits on all annotations.


## 01 — Drive environment and audited input registry


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
DATA=next((p for p in [DRIVE/'Colab Notebooks/Musanga/Final/Data',DRIVE/'Musanga/Final/Data'] if p.exists()),None)
assert DATA is not None,'Final/Data directory not found'
AUDIT=DATA.parent/'Outputs/V3/00_input_audit'
OUT=DATA.parent/'Outputs/V3/01_reference_data';OUT.mkdir(parents=True,exist_ok=True)
manifest_path=AUDIT/'00_input_manifest.json'
assert manifest_path.exists(),'Run Notebook 00 through Block 10 first'
A=json.loads(manifest_path.read_text())
MOSAICS={r['file']:r for r in A['mosaics']}
TREE_FILES={site:Path(p) for site,p in A['shared_annotations'].items()}
ROI_FILE=Path(A['roi'])
CIRCLE_FILE=Path(A['exhaustive_annotations']['complete_roi_boundaries'])
EXH_FILE=Path(A['exhaustive_annotations']['positive_polygons'])
SITES=['Baego','MOS','Yangambi','Yoko']
FOLDS=3;PATCH=512;BUFFER_M=15.;SEED=20261015
for p in [ROI_FILE,CIRCLE_FILE,EXH_FILE,*TREE_FILES.values(),*[Path(r['path']) for r in MOSAICS.values()]]:
    assert p.exists(),p
print('Found',len(MOSAICS),'RGB mosaics and',len(TREE_FILES),'shared tree annotation files')
print('Read-only input:',DATA,'\nWrites:',OUT)


## 02 — Repair geometry individually, with traceable quarantines

Invalid geometries repaired in memory if relative area change <= 5%; otherwise excluded individually, never all polygons from the same file. Review quarantines before relying on the final outputs.


In [ ]:
REPAIRS=[]
def parts(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in parts(q)]
    return []
def load_polygons(path):
    raw=gpd.read_file(path)
    assert raw.crs is not None,path
    rows=[];geoms=[]
    for idx,g in enumerate(raw.geometry):
        status='OK';delta=0.
        if g is None or g.is_empty:status='EMPTY'
        elif not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon'):
            try:
                fixed=unary_union(parts(make_valid(g)))
                delta=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if not fixed.is_valid or fixed.is_empty or delta>.05:status='QUARANTINE'
                else:g=fixed;status='REPAIRED'
            except Exception:status='QUARANTINE'
        REPAIRS.append(dict(file=path.name,row=idx,status=status,area_change=delta))
        if status in ('OK','REPAIRED'):rows.append(idx);geoms.append(g)
    output=raw.iloc[rows].copy().reset_index(drop=True)
    output.geometry=geoms
    output['_row_original']=rows
    print(path.name,':',len(output),'kept,',len(raw)-len(output),'quarantined')
    return output
ROI=load_polygons(ROI_FILE);CIRCLES=load_polygons(CIRCLE_FILE)
EXH=load_polygons(EXH_FILE)
TREES={site:load_polygons(path) for site,path in TREE_FILES.items()}
pd.DataFrame(REPAIRS).to_csv(OUT/'01_geometry_repairs.csv',index=False)
assert ROI.geometry.is_valid.all()
for site,g in TREES.items():
    assert 'Class_ID' in g.columns,site
    cls=pd.to_numeric(g.Class_ID,errors='coerce')
    g['_label']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1)).astype('int8')
    print(site,':',int((g._label==1).sum()),'Musanga /',int((g._label==0).sum()),'known negatives')
# As in Notebook 00: candidate circles with no Musanga are not assumed exhaustive.
exh_union=unary_union(EXH.to_crs(CIRCLES.crs).geometry.to_list())
verified=[i for i,g in enumerate(CIRCLES.geometry) if g.intersection(exh_union).area>1]
CIRCLES=CIRCLES.iloc[verified].copy().reset_index(drop=True)
CIRCLES['circle_id']=CIRCLES['_row_original'].astype(int)
assert len(CIRCLES)==22,f'Expected 22 verified circles; got {len(CIRCLES)}'
print('Verified exhaustive circles:',list(CIRCLES.circle_id))


## 03 — Six mosaics nested within four forest groups; hard general ROI


In [ ]:
REG=[];GEO={}
for name,info in MOSAICS.items():
    with rasterio.open(info['path']) as ds:
        assert ds.crs and ds.crs.is_projected and ds.count>=3,name
        footprint=box(*ds.bounds)
        roi=ROI.to_crs(ds.crs)
        eligible=unary_union(roi.geometry.to_list()).intersection(footprint)
        REG.append(dict(mosaic=name,site=info['site'],roi_overlap_m2=float(eligible.area),
                        width=ds.width,height=ds.height,pixel_m=abs(ds.res[0]),crs=str(ds.crs)))
        GEO[name]=dict(transform=ds.transform,crs=ds.crs,width=ds.width,height=ds.height,
                       footprint=footprint,roi=roi,site=info['site'])
REG=pd.DataFrame(REG);display(REG)
REG.to_csv(OUT/'01_mosaic_registry.csv',index=False)
assert (REG.roi_overlap_m2>0).all(),'At least one mosaic lacks ROI coverage: resolve ROI.shp before continuing'
